# Chapter 1: LLM Architecture — The Brain of Every Agent

> **Book:** Agentic AI Engineering  
> **Part:** 1 — Foundations  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# pip install transformers torch tiktoken matplotlib numpy

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")
if TEST_MODE:
    print("Running in TEST_MODE — API calls and heavy model loads will be skipped.")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:

- Understand the **decoder-only transformer** architecture that powers GPT-4, Claude, and Llama
- Know how **tokenization (BPE)** converts medical text into token IDs
- Visualize **self-attention weights** to see what the model "focuses on"
- Understand the **KV cache** and why it matters for agent latency
- Build a reusable **token counting utility** for prompt engineering

**Why it matters for agents:** Every agent in the Guardian Angel System (GAS) is powered by an LLM. Understanding the architecture helps you design better prompts, manage context windows, and predict latency.

---
## 📖 2. Theory Recap

### 2.1 Decoder-Only Transformer

Modern LLMs (GPT, Llama, Claude) use a **decoder-only** architecture:

```
Input Tokens → Embedding → [N × Transformer Block] → LM Head → Next Token Probabilities

Transformer Block:
  x → LayerNorm → Multi-Head Self-Attention → + residual
    → LayerNorm → Feed-Forward Network (MLP) → + residual
```

Key properties:
- **Causal masking**: each token can only attend to previous tokens
- **Autoregressive**: generates one token at a time
- **Residual connections**: enable training very deep networks

### 2.2 Tokenization (BPE)

**Byte-Pair Encoding (BPE)** builds a vocabulary by iteratively merging the most frequent character pairs:

1. Start with character-level vocabulary
2. Count all adjacent pairs
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary size is reached

Medical text tokenizes differently than general text — `"mg/dL"` might be 3-4 tokens.

### 2.3 Self-Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q (Query)**: what am I looking for?
- **K (Key)**: what do I contain?
- **V (Value)**: what do I output?
- **Multi-head**: run H attention heads in parallel, concatenate

### 2.4 KV Cache

During generation, Keys and Values for previous tokens don't change. The **KV cache** stores them so we don't recompute:

- **Without cache**: O(n²) computation per token
- **With cache**: O(n) computation per token (only compute new token's Q)
- **Memory cost**: `2 × n_layers × n_heads × head_dim × seq_len × bytes_per_param`

### 2.5 Why This Matters for Agents

| Concept | Agent Impact |
|---------|-------------|
| Context window | Max prompt + history length |
| Tokenization | Prompt engineering efficiency |
| KV cache | Latency for long conversations |
| Attention | What the model "reads" in your prompt |

---
## 🔨 3. Build It

### 3.1 Tokenization Demo

In [ ]:
import tiktoken
import json

# Load the cl100k_base encoding (used by GPT-4, Claude, etc.)
enc = tiktoken.get_encoding("cl100k_base")

# Medical text examples
medical_texts = [
    "Patient glucose is 245 mg/dL, administer 2 units insulin",
    "HbA1c: 8.2%, fasting glucose 180 mg/dL, post-prandial 240 mg/dL",
    "Continuous glucose monitor (CGM) alert: hypoglycemia detected, glucose < 70 mg/dL",
    "Insulin pump basal rate: 0.8 U/hr, correction factor: 1:50",
]

print("=" * 70)
print("TOKENIZATION DEMO — Medical Text")
print("=" * 70)

for text in medical_texts:
    tokens = enc.encode(text)
    decoded_tokens = [enc.decode([t]) for t in tokens]
    print(f"\nText: {text}")
    print(f"Token count: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Tokens: {decoded_tokens}")
    print("-" * 50)

In [ ]:
# ── Token Counting Utility ──────────────────────────────────────────────────

def count_tokens(text: str, model: str = "gpt-4") -> dict:
    """
    Count tokens in text for a given model.
    
    Args:
        text: Input text string
        model: Model name (affects encoding)
    
    Returns:
        dict with token_count, char_count, tokens_per_char, estimated_cost_usd
    """
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    
    tokens = enc.encode(text)
    token_count = len(tokens)
    char_count = len(text)
    
    # Approximate cost (GPT-4 input pricing as of 2024)
    cost_per_1k_tokens = 0.01  # $0.01 per 1K input tokens
    estimated_cost = (token_count / 1000) * cost_per_1k_tokens
    
    return {
        "token_count": token_count,
        "char_count": char_count,
        "tokens_per_char": round(token_count / max(char_count, 1), 3),
        "estimated_cost_usd": round(estimated_cost, 6),
        "fits_in_4k_context": token_count <= 4096,
        "fits_in_8k_context": token_count <= 8192,
        "fits_in_128k_context": token_count <= 131072,
    }


# Test with a GAS system prompt
gas_system_prompt = """
You are the Monitor Agent in the Guardian Angel System (GAS), a multi-agent 
system for diabetic patient supervision. Your role is to:
1. Continuously monitor CGM (Continuous Glucose Monitor) readings
2. Detect anomalies: hypoglycemia (<70 mg/dL) and hyperglycemia (>180 mg/dL)
3. Alert the Advisor Agent when intervention may be needed
4. Log all readings with timestamps for the Memory Agent

Current patient profile:
- Name: Patient A
- Diabetes type: Type 1
- Target range: 70-180 mg/dL
- Insulin pump: Yes (basal rate 0.8 U/hr)
"""

result = count_tokens(gas_system_prompt)
print("Token Analysis for GAS Monitor Agent System Prompt:")
print(json.dumps(result, indent=2))

### 3.2 Attention Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

if not TEST_MODE:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    print("Loading distilgpt2 (small model for demo)...")
    model_name = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        output_attentions=True,
        attn_implementation="eager"  # needed for attention output
    )
    model.eval()
    
    # Medical text for attention visualization
    text = "Patient glucose is 245 mg/dL administer insulin"
    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get attention from last layer, first head
    # Shape: (batch, heads, seq_len, seq_len)
    attention = outputs.attentions[-1][0, 0].numpy()  # last layer, head 0
    
else:
    # Mock attention weights for TEST_MODE
    print("TEST_MODE: Using mock attention weights")
    tokens = ["Patient", "Ġglucose", "Ġis", "Ġ245", "Ġmg", "/", "dL", "Ġadminister", "Ġinsulin"]
    n = len(tokens)
    # Create a realistic-looking causal attention pattern
    attention = np.zeros((n, n))
    for i in range(n):
        weights = np.random.dirichlet(np.ones(i + 1) * 2)
        attention[i, :i+1] = weights
    # Make glucose→insulin connection strong
    attention[8, 1] = 0.4  # insulin attends to glucose
    attention[8, 3] = 0.3  # insulin attends to 245

# Plot attention heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full attention matrix
ax1 = axes[0]
im1 = ax1.imshow(attention, cmap='Blues', aspect='auto', vmin=0, vmax=attention.max())
ax1.set_xticks(range(len(tokens)))
ax1.set_yticks(range(len(tokens)))
ax1.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(tokens, fontsize=9)
ax1.set_title("Self-Attention Weights\n(Last Layer, Head 0)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Key (attended to)")
ax1.set_ylabel("Query (attending from)")
plt.colorbar(im1, ax=ax1, label='Attention Weight')

# Add value annotations
for i in range(len(tokens)):
    for j in range(len(tokens)):
        if attention[i, j] > 0.1:
            ax1.text(j, i, f'{attention[i,j]:.2f}', ha='center', va='center', 
                    fontsize=7, color='white' if attention[i,j] > 0.3 else 'black')

# Attention from 'insulin' token (last token)
ax2 = axes[1]
last_token_attention = attention[-1]
colors = ['#e74c3c' if v > 0.2 else '#3498db' for v in last_token_attention]
bars = ax2.bar(range(len(tokens)), last_token_attention, color=colors)
ax2.set_xticks(range(len(tokens)))
ax2.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
ax2.set_title(f'Attention from "{tokens[-1]}" token\n(What does "insulin" attend to?)', 
              fontsize=12, fontweight='bold')
ax2.set_ylabel('Attention Weight')
ax2.set_xlabel('Token')
ax2.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='0.1 threshold')
ax2.legend()

plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey insight: The 'insulin' token attends strongly to 'glucose' and '245'")
print("This shows the model understands the causal relationship: high glucose → insulin")

### 3.3 KV Cache Demo — Speedup Measurement

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

if not TEST_MODE:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    # Use the already-loaded model
    tokenizer_kv = AutoTokenizer.from_pretrained("distilgpt2")
    model_kv = AutoModelForCausalLM.from_pretrained("distilgpt2")
    model_kv.eval()
    
    prompt = "Patient glucose is 245 mg/dL. The recommended action is"
    inputs = tokenizer_kv(prompt, return_tensors="pt")
    
    # Measure generation WITH KV cache (default)
    times_with_cache = []
    for _ in range(5):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model_kv.generate(**inputs, max_new_tokens=20, use_cache=True, do_sample=False)
        times_with_cache.append(time.perf_counter() - start)
    
    # Measure generation WITHOUT KV cache
    times_without_cache = []
    for _ in range(5):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model_kv.generate(**inputs, max_new_tokens=20, use_cache=False, do_sample=False)
        times_without_cache.append(time.perf_counter() - start)
    
    avg_with = np.mean(times_with_cache)
    avg_without = np.mean(times_without_cache)
    speedup = avg_without / avg_with
    
else:
    # Mock results for TEST_MODE
    print("TEST_MODE: Using simulated KV cache benchmark results")
    avg_with = 0.42
    avg_without = 1.87
    speedup = avg_without / avg_with
    times_with_cache = [0.40, 0.42, 0.41, 0.43, 0.44]
    times_without_cache = [1.85, 1.88, 1.90, 1.84, 1.88]

print("KV Cache Benchmark Results")
print("=" * 40)
print(f"With KV cache:    {avg_with*1000:.1f} ms")
print(f"Without KV cache: {avg_without*1000:.1f} ms")
print(f"Speedup:          {speedup:.2f}x")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
ax1.bar(['With KV Cache', 'Without KV Cache'], 
        [avg_with * 1000, avg_without * 1000],
        color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
ax1.set_ylabel('Generation Time (ms)')
ax1.set_title('KV Cache Speedup\n(20 new tokens, distilgpt2)', fontweight='bold')
ax1.text(0, avg_with * 1000 + 10, f'{avg_with*1000:.0f}ms', ha='center', fontweight='bold')
ax1.text(1, avg_without * 1000 + 10, f'{avg_without*1000:.0f}ms', ha='center', fontweight='bold')

# Theoretical scaling
seq_lengths = np.arange(10, 500, 10)
# Without cache: O(n^2) per step, n steps → O(n^3)
# With cache: O(n) per step, n steps → O(n^2)
time_no_cache = (seq_lengths ** 2) / 1000  # normalized
time_with_cache = seq_lengths / 100  # normalized

ax2.plot(seq_lengths, time_no_cache, 'r-', label='Without KV Cache O(n²)', linewidth=2)
ax2.plot(seq_lengths, time_with_cache, 'g-', label='With KV Cache O(n)', linewidth=2)
ax2.set_xlabel('Sequence Length (tokens)')
ax2.set_ylabel('Relative Computation Cost')
ax2.set_title('KV Cache Scaling Advantage', fontweight='bold')
ax2.legend()
ax2.set_xlim(0, 500)
ax2.fill_between(seq_lengths, time_with_cache, time_no_cache, alpha=0.1, color='green',
                 label='Savings')

plt.tight_layout()
plt.savefig('kv_cache_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💡 For GAS agents with 4K context windows, KV cache provides ~{speedup:.1f}x speedup")
print("   This is critical for real-time glucose monitoring responses!")

### 3.4 Token Counting for GAS Prompt Engineering

In [ ]:
import tiktoken
import matplotlib.pyplot as plt

enc = tiktoken.get_encoding("cl100k_base")

# GAS Agent prompts
gas_prompts = {
    "Monitor Agent": """
        You are the Monitor Agent. Analyze CGM readings and detect anomalies.
        Current reading: {glucose} mg/dL at {timestamp}.
        Previous readings (last 2 hours): {history}
        Alert if: glucose < 70 (hypoglycemia) or > 180 (hyperglycemia)
        Respond with: status, trend, alert_level (0-3)
    """,
    "Advisor Agent": """
        You are the Advisor Agent. Given patient state, recommend action.
        Patient: {patient_profile}
        Current glucose: {glucose} mg/dL
        Recent trend: {trend}
        Active insulin: {iob} units on board
        Meal context: {meal_info}
        Recommend: insulin dose, carb intake, or no action. Explain reasoning.
    """,
    "Safety Critic": """
        You are the Safety Critic. Evaluate the proposed action for safety.
        Proposed action: {action}
        Patient state: glucose={glucose}, IOB={iob}, trend={trend}
        Check: Is the action safe? Could it cause hypoglycemia?
        Output: safety_score (0-1), approved (true/false), concerns
    """,
    "Memory Agent": """
        You are the Memory Agent. Retrieve relevant patient history.
        Query: {query}
        Available history: {history_summary}
        Return: relevant_events, patterns, recommendations_history
    """,
    "Communicator Agent": """
        You are the Communicator Agent. Translate medical recommendations to patient-friendly language.
        Recommendation: {recommendation}
        Patient literacy level: {literacy_level}
        Urgency: {urgency}
        Output: patient_message, action_steps, when_to_call_doctor
    """
}

print("GAS Agent Token Budget Analysis")
print("=" * 50)
print(f"{'Agent':<25} {'Tokens':>8} {'% of 4K':>10} {'% of 8K':>10}")
print("-" * 55)

token_counts = {}
for agent, prompt in gas_prompts.items():
    count = len(enc.encode(prompt))
    token_counts[agent] = count
    pct_4k = count / 4096 * 100
    pct_8k = count / 8192 * 100
    print(f"{agent:<25} {count:>8} {pct_4k:>9.1f}% {pct_8k:>9.1f}%")

total = sum(token_counts.values())
print("-" * 55)
print(f"{'TOTAL (all agents)':<25} {total:>8}")
print(f"\nRemaining budget per agent (8K context):")
for agent, count in token_counts.items():
    remaining = 8192 - count
    print(f"  {agent}: {remaining} tokens for conversation history")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
agents = list(token_counts.keys())
counts = list(token_counts.values())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

bars = ax.barh(agents, counts, color=colors, alpha=0.8, edgecolor='black')
ax.axvline(x=4096, color='red', linestyle='--', alpha=0.7, label='4K context limit')
ax.axvline(x=8192, color='orange', linestyle='--', alpha=0.7, label='8K context limit')
ax.set_xlabel('Token Count')
ax.set_title('GAS Agent System Prompt Token Usage', fontweight='bold')
ax.legend()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, 
            str(count), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('gas_token_budget.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏥 4. GAS Connection

### How LLM Architecture Powers the Guardian Angel System

```
                    GUARDIAN ANGEL SYSTEM (GAS)
                    
  CGM Device ──→ [Supervisor]      ←── LLM Brain (orchestrator / router)
                      │
                      ↓
              [Monitor Agent]    ←── LLM Brain (anomaly detection)
                      │
                      ↓
               [Advisor Agent]   ←── LLM Brain (RAG + memory recommendations)
                      │
                      ↓
              [Safety Critic]    ←── LLM Brain (reward-model validation)
                      │
                      ↓
          [Caregiver Notifier]   ←── LLM Brain (human escalation via A2A)
                      │
                      ↓
                  Patient 👤
```

**Key architectural insights for GAS:**

1. **Tokenization**: CGM readings like `"glucose=245 mg/dL"` become ~5 tokens. A 2-hour history (24 readings) ≈ 120 tokens — fits easily in any context window.

2. **Attention**: The Advisor Agent's attention mechanism naturally focuses on the most recent glucose readings and the patient's insulin-on-board (IOB) — the most safety-critical values.

3. **KV Cache**: For the Monitor Agent running every 5 minutes, the KV cache means the system prompt is only processed once. Subsequent readings are fast O(n) operations.

4. **Context Window Planning**:
   - System prompt: ~100 tokens
   - 24-hour glucose history: ~500 tokens  
   - Current reading + context: ~50 tokens
   - **Total: ~650 tokens** — well within any model's context window

In [ ]:
# GAS Context Window Calculator
def calculate_gas_context_needs(
    history_hours: int = 24,
    readings_per_hour: int = 12,  # CGM reads every 5 min
    tokens_per_reading: int = 8,
    system_prompt_tokens: int = 150,
    response_tokens: int = 200
) -> dict:
    """Calculate context window requirements for GAS agents."""
    history_readings = history_hours * readings_per_hour
    history_tokens = history_readings * tokens_per_reading
    total_input = system_prompt_tokens + history_tokens
    total_with_response = total_input + response_tokens
    
    return {
        "history_readings": history_readings,
        "history_tokens": history_tokens,
        "system_prompt_tokens": system_prompt_tokens,
        "total_input_tokens": total_input,
        "total_with_response": total_with_response,
        "fits_in_4k": total_with_response <= 4096,
        "fits_in_8k": total_with_response <= 8192,
        "recommended_context": "4K" if total_with_response <= 4096 else "8K" if total_with_response <= 8192 else "32K+"
    }

print("GAS Context Window Requirements")
print("=" * 50)
for hours in [1, 6, 24, 72]:
    result = calculate_gas_context_needs(history_hours=hours)
    print(f"\n{hours}h history ({result['history_readings']} readings):")
    print(f"  History tokens:  {result['history_tokens']:,}")
    print(f"  Total tokens:    {result['total_with_response']:,}")
    print(f"  Recommended:     {result['recommended_context']} context")
    print(f"  Fits in 4K:      {'✅' if result['fits_in_4k'] else '❌'}")

---
## ✅ 5. Checkpoint

### What We Built

| Component | Description | Reusable? |
|-----------|-------------|----------|
| `count_tokens()` | Token counting utility with cost estimation | ✅ Yes |
| Attention visualizer | Heatmap of self-attention weights | ✅ Yes |
| KV cache benchmark | Speedup measurement with/without cache | ✅ Yes |
| `calculate_gas_context_needs()` | Context window planner for GAS | ✅ Yes |

### Key Takeaways

1. **Tokenization is not character-splitting** — medical terms like `mg/dL` become multiple tokens
2. **Attention shows what the model focuses on** — `insulin` attends to `glucose` and `245`
3. **KV cache provides ~4x speedup** for generation — critical for real-time GAS monitoring
4. **GAS fits comfortably in 4K context** for 1-6 hour histories

### What Comes Next

**Chapter 2: Systems Foundations** — Now that we understand the LLM brain, we'll look at the hardware it runs on: GPU memory, batching strategies, and how to run 5 GAS agents simultaneously without running out of VRAM.

```
Chapter 1 ✅ → Chapter 2 (GPU Memory & Batching) → Chapter 3 (RL Foundations)
```